In [3]:
import numpy as np 
import pandas as pd
import torch 
import torch.nn as nn
device = torch.device('cuda')
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import optuna
from torch import optim as optim

In [4]:
torch.cuda.is_available()

True

In [5]:
train = pd.read_csv('https://raw.githubusercontent.com/guilhermedom/cnn-fashion-mnist/main/data/raw/fashion-mnist-train.zip',compression='zip')
test = pd.read_csv('https://raw.githubusercontent.com/guilhermedom/cnn-fashion-mnist/main/data/raw/fashion-mnist-test.zip',compression='zip')

In [6]:
train.shape,test.shape

((60000, 785), (10000, 785))

In [7]:
train.sample()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
6719,2,0,0,0,0,0,0,0,1,0,...,3,0,24,169,175,105,0,0,0,0


In [8]:
df = pd.concat([train,test])

In [9]:
X = df.iloc[:,1:]
y = df.iloc[:,0]

In [10]:
X = X/255.0

In [11]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return self.features[idx],self.labels[idx]

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)

In [13]:
X_train = torch.tensor(X_train.values,dtype=torch.float32)
y_train = torch.tensor(y_train.values,dtype=torch.long)
X_test = torch.tensor(X_test.values,dtype=torch.float32)
y_test = torch.tensor(y_test.values,dtype=torch.long)

In [14]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [29]:
class ANN(nn.Module):
    def __init__(self,input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate,activation):
        super().__init__()
        layers = []

        if activation == "relu":
            act = nn.ReLU()
        elif activation == "gelu":
            act = nn.GELU()
        else:
            act = nn.Tanh()

        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim,neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(act)
            layers.append(nn.Dropout(p=dropout_rate))
            input_dim = neurons_per_layer
        layers.append(nn.Linear(neurons_per_layer,output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [30]:
def objective(trial):
    #next hyperparameter value from search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers",1,5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer",8,128,step=8)
    epochs = trial.suggest_int("epochs",10,50,step=10)
    learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1,log=True)
    dropout_rate = trial.suggest_float("dropout_rate",.1,.5,step=.1)
    batch_size = trial.suggest_categorical("batch_size",[32, 64, 128, 256, 512])
    optimizer_name = trial.suggest_categorical("optimizer_name",['Adam','SGD','RMSprop'])
    weight_decay = trial.suggest_float("weight_decay",1e-5,1e-3,log=True)
    activation = trial.suggest_categorical("activation", ["relu","gelu","tanh"])

    train_loader = DataLoader(train_dataset,shuffle=True,pin_memory=True,batch_size=batch_size)
    test_loader = DataLoader(test_dataset,shuffle=False,pin_memory=True,batch_size=batch_size)

    #model init
    input_dim = 784
    output_dim = 10

    model = ANN(input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate,activation)
    model.to(device)

    #parameter init

    #optimizer selection
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)

    #training loop
    for epoch in range(epochs):
        model.train()
        for batch_feature,batch_label in train_loader:

            batch_feature = batch_feature.to(device)
            batch_label = batch_label.to(device)

            outputs = model(batch_feature)
            loss = criterion(outputs,batch_label)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    #evaluation
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for batch_feature, batch_label in test_loader:

            batch_feature = batch_feature.to(device)
            batch_label = batch_label.to(device)

            outputs = model(batch_feature)

            _, predicted = torch.max(outputs, 1)

            total += batch_feature.shape[0]
            correct += (predicted == batch_label).sum().item()

        accuracy = correct / total
    return accuracy

In [35]:
study = optuna.create_study(direction='maximize',study_name='gandu')

[I 2026-03-16 21:37:01,363] A new study created in memory with name: gandu


In [ ]:
study.optimize(objective,n_trials=50)

[I 2026-03-16 21:37:55,287] Trial 0 finished with value: 0.8368571428571429 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 16, 'epochs': 30, 'learning_rate': 0.003136671456676209, 'dropout_rate': 0.4, 'batch_size': 256, 'optimizer_name': 'SGD', 'weight_decay': 0.0001102919685194794, 'activation': 'tanh'}. Best is trial 0 with value: 0.8368571428571429.
[I 2026-03-16 21:41:00,064] Trial 1 finished with value: 0.7625 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 80, 'epochs': 50, 'learning_rate': 4.1308027711733216e-05, 'dropout_rate': 0.2, 'batch_size': 64, 'optimizer_name': 'SGD', 'weight_decay': 9.050711118237923e-05, 'activation': 'gelu'}. Best is trial 0 with value: 0.8368571428571429.
[I 2026-03-16 21:42:58,153] Trial 2 finished with value: 0.8131428571428572 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 128, 'epochs': 20, 'learning_rate': 0.008587152770200073, 'dropout_rate': 0.5, 'batch_size': 32, 'optimizer_name': 'RMSprop', 'weight

In [ ]:
study.best_params

{'num_hidden_layers': 3,
 'neurons_per_layer': 72,
 'epochs': 30,
 'learning_rate': 2.4324575053059678e-05,
 'dropout_rate': 0.1,
 'batch_size': 32,
 'optimizer_name': 'Adam',
 'weight_decay': 6.148876759911207e-05,
 'activation': 'gelu'}

In [ ]:
study.best_value

0.886